# WGAN / WGAN-GP — replace JS divergence with the Wasserstein distance

> Tutorial pair for [`wgan.py`](wgan.py).

## 1. Intuition
When the real and fake distributions barely overlap (always true early in
training), the Jensen–Shannon divergence the vanilla GAN minimizes is locally
*constant* — the generator gets no gradient. The **Wasserstein-1 (earth-mover)
distance** measures the cost of *moving mass* from one distribution to the other
and varies smoothly even for disjoint supports, so it always points the generator
somewhere useful.

## 2. Concept (the slide)
- The discriminator becomes a **critic** $f$: an unbounded real-valued score, not
  a probability (no sigmoid).
- The critic must be **1-Lipschitz**. Two enforcements:
  - **Weight clipping** (original WGAN): box-clip all critic weights to
    $[-c, c]$. Crude but works; pair with RMSProp.
  - **Gradient penalty** (WGAN-GP): softly pin $\|\nabla f\|_2$ to 1 on points
    interpolated between real and fake. Use Adam, no BatchNorm in the critic.
- Train the critic `n_critic` times per generator step so it stays near optimal.

## 3. Math derivation — Wasserstein distance and its dual

The Wasserstein-1 distance between $p_{\text{data}}$ and $p_g$ is
$$W(p_{\text{data}},p_g)=\inf_{\gamma\in\Pi(p_{\text{data}},p_g)}
 \mathbb E_{(x,y)\sim\gamma}\,\|x-y\|,$$
an infimum over couplings $\gamma$ with the right marginals. This primal is
intractable, but **Kantorovich–Rubinstein duality** rewrites it as a maximization
over 1-Lipschitz functions:
$$W(p_{\text{data}},p_g)=\sup_{\|f\|_L\le 1}
 \mathbb E_{x\sim p_{\text{data}}}[f(x)]-\mathbb E_{x\sim p_g}[f(x)].$$

The critic $f$ approximates the optimal witness function. This gives the WGAN
objective (a min–max, like the GAN, but with a *different* value function):
$$\min_G\max_{\|f\|_L\le 1}\;
 \mathbb E_{x\sim p_{\text{data}}}[f(x)]-\mathbb E_{z\sim p_z}[f(G(z))].$$

**Why this beats JS.** For disjoint supports the JS divergence is the constant
$\log 2$, so $\nabla_G\,\mathrm{JSD}=0$; but $W$ scales with the *distance* between
supports, giving a non-vanishing $\nabla_G W$ everywhere.

**Enforcing $\|f\|_L\le 1$.**
- *Weight clipping:* constrain weights to $[-c,c]$ so $f$ is Lipschitz — but it
  biases the critic toward simple functions and can cause exploding/vanishing
  gradients.
- *Gradient penalty (WGAN-GP):* a 1-Lipschitz $f$ has $\|\nabla_x f(x)\|_2\le 1$,
  and the optimal critic has unit-norm gradient almost everywhere on the optimal
  coupling's support. So add
  $$\lambda\,\mathbb E_{\hat x}\big[(\|\nabla_{\hat x} f(\hat x)\|_2-1)^2\big],
  \qquad \hat x=\epsilon\,x+(1-\epsilon)\,G(z),\ \epsilon\sim U[0,1],$$
  penalizing deviation of the gradient norm from 1 on the lines between real and
  fake points. The reported $\mathbb E[f(\text{real})]-\mathbb E[f(\text{fake})]$
  is a running **estimate of $W$**.

## 4. Generator / key component

In [ ]:
# ===== actual implementation from wgan.py =====
from __future__ import annotations

import numpy as np

import torch

import torch.nn as nn

SEED = 0

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def make_ring(n: int = 2000, k: int = 8, r: float = 2.0, seed: int = SEED) -> np.ndarray:
    rng = np.random.default_rng(seed)
    ang = 2 * np.pi * rng.integers(0, k, n) / k
    centers = np.c_[r * np.cos(ang), r * np.sin(ang)]
    return (centers + 0.1 * rng.normal(size=(n, 2))).astype(np.float32)

class Generator(nn.Module):
    def __init__(self, noise_dim: int = 8, data_dim: int = 2, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, hidden), nn.ReLU(True),
            nn.Linear(hidden, hidden), nn.ReLU(True),
            nn.Linear(hidden, data_dim))

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)

class Critic(nn.Module):
    def __init__(self, data_dim: int = 2, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

## 5. Trainer / losses

In [ ]:
# ===== actual implementation from wgan.py =====
class WGANTorch:
    """WGAN / WGAN-GP. clip='weight' (clipping) or clip='gp' (gradient penalty)."""

    def __init__(self, noise_dim: int = 8, data_dim: int = 2, clip: str = "gp",
                 lr: float = 1e-4, clip_val: float = 0.01, gp_lambda: float = 10.0,
                 n_critic: int = 5):
        torch.manual_seed(SEED)
        assert clip in ("weight", "gp")
        self.dev = get_device()
        self.noise_dim, self.clip = noise_dim, clip
        self.clip_val, self.gp_lambda, self.n_critic = clip_val, gp_lambda, n_critic
        self.G = Generator(noise_dim, data_dim).to(self.dev)
        self.D = Critic(data_dim).to(self.dev)
        if clip == "weight":
            # The WGAN paper recommends RMSProp (Adam was unstable with clipping).
            self.optG = torch.optim.RMSprop(self.G.parameters(), lr=lr)
            self.optD = torch.optim.RMSprop(self.D.parameters(), lr=lr)
        else:
            self.optG = torch.optim.Adam(self.G.parameters(), lr=lr, betas=(0.5, 0.9))
            self.optD = torch.optim.Adam(self.D.parameters(), lr=lr, betas=(0.5, 0.9))

    def _gradient_penalty(self, real: torch.Tensor, fake: torch.Tensor) -> torch.Tensor:
        """E[(||grad_x D(x_hat)||_2 - 1)^2] on points x_hat on lines real<->fake."""
        b = real.size(0)
        eps = torch.rand(b, 1, device=self.dev)
        xhat = (eps * real + (1 - eps) * fake).requires_grad_(True)
        score = self.D(xhat)
        grad = torch.autograd.grad(
            outputs=score, inputs=xhat,
            grad_outputs=torch.ones_like(score),
            create_graph=True, retain_graph=True)[0]
        gnorm = grad.norm(2, dim=1)
        return ((gnorm - 1.0) ** 2).mean()

    def fit(self, real: np.ndarray, steps: int = 1500, batch: int = 128):
        real = torch.as_tensor(real, dtype=torch.float32, device=self.dev)
        self.d_hist, self.w_hist = [], []  # critic loss; Wasserstein estimate
        for _ in range(steps):
            # --- train the critic n_critic times ---
            for _ in range(self.n_critic):
                idx = torch.randint(0, len(real), (batch,), device=self.dev)
                x = real[idx]
                z = torch.randn(batch, self.noise_dim, device=self.dev)
                fake = self.G(z).detach()
                # Critic maximizes E[D(real)] - E[D(fake)]  => minimize negation.
                d_real = self.D(x).mean()
                d_fake = self.D(fake).mean()
                lossD = -(d_real - d_fake)
                if self.clip == "gp":
                    lossD = lossD + self.gp_lambda * self._gradient_penalty(x, fake)
                self.optD.zero_grad(); lossD.backward(); self.optD.step()
                if self.clip == "weight":
                    # crude 1-Lipschitz enforcement: box-clip every weight
                    for p in self.D.parameters():
                        p.data.clamp_(-self.clip_val, self.clip_val)
                self.w_hist.append((d_real - d_fake).item())  # Wasserstein estimate
            # --- train the generator once: maximize E[D(G(z))] ---
            z = torch.randn(batch, self.noise_dim, device=self.dev)
            lossG = -self.D(self.G(z)).mean()
            self.optG.zero_grad(); lossG.backward(); self.optG.step()
            self.d_hist.append(lossD.item())
        return self

    @torch.no_grad()
    def generate(self, n: int) -> np.ndarray:
        z = torch.randn(n, self.noise_dim, device=self.dev)
        return self.G(z).cpu().numpy()

def _coverage(fake: np.ndarray, k: int = 8) -> int:
    modes = np.arctan2(fake[:, 1], fake[:, 0])
    return len(np.unique(np.round(modes / (2 * np.pi / k)).astype(int) % k))

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    real = make_ring(2000)
    print(f"real mean={real.mean(0).round(2)} std={real.std(0).round(2)}")

    for mode in ("gp", "weight"):
        gan = WGANTorch(clip=mode).fit(real, steps=600, batch=128)
        fake = gan.generate(800)
        w0 = np.mean(gan.w_hist[:100]); w1 = np.mean(gan.w_hist[-100:])
        print(f"[{mode:6s}] Wasserstein estimate {w0:.3f} -> {w1:.3f} "
              f"(should rise then plateau); covered {_coverage(fake)}/8 modes; "
              f"fake std={fake.std(0).round(2)}")

## 6. Train

In [ ]:
demo()

## 7. Visualization

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import wgan as M

real = M.make_ring(2000)
gan = M.WGANTorch(clip="gp").fit(real, steps=600, batch=128)
fake = gan.generate(1000)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(real[:, 0], real[:, 1], s=6, alpha=.3, label="real")
ax[0].scatter(fake[:, 0], fake[:, 1], s=6, alpha=.5, color="r", label="fake")
ax[0].set_title("Real vs generated (WGAN-GP)"); ax[0].legend(); ax[0].set_aspect("equal")
ax[1].plot(gan.w_hist, alpha=.6)
ax[1].set_xlabel("critic step"); ax[1].set_ylabel(r"$E[f(real)]-E[f(fake)]$")
ax[1].set_title("Wasserstein estimate (rises then plateaus)")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- WGAN swaps the JS-divergence value function for the **Wasserstein distance** via
  Kantorovich–Rubinstein duality, giving usable gradients for disjoint supports.
- The critic must be **1-Lipschitz**; gradient penalty is smoother and more robust
  than weight clipping.
- Pitfalls: **no BatchNorm in the GP critic** (it breaks the per-sample gradient
  penalty); too-large clip values destroy the Lipschitz constraint; too-few
  `n_critic` steps leave the $W$-estimate unreliable.